# Fire impacts simulations

This notebook demonstrates the fire impacts simulation modules.

It is assumed that you have a `FireImpactsProject` populated with all pre-processed data for the catchment.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

from fire_impacts.sim import rusle, aggregate_rainfall_data, debris
from fire_impacts.stochastic.rainfall import get_rainfall_replicates
from fire_impacts import FireImpactsProject

import matplotlib.pyplot as plt

## Load project

In [ ]:
proj = FireImpactsProject('.',exist_ok=True)

In [ ]:
proj.catchments

**Note:** You can capture multiple study areas / catchments within a single `FireImpactsProject` directory structure. Here we will assume you have one, so we will take the first (presumed only)

In [ ]:
catchment_name = proj.catchments[0]
catchment_name

## Rainfall data

Both the erosion (RUSLE) and debris flow modules rely on subdaily rainfall data. The erosion modules, demonstrated here, relies on 30 minute data, while the debris flow relies on 12 minute data.

Furthermore, it is encouraged that both modules be run stochastically, using multiple rainfall replicates.

The library provides functionality for processing stochastic rainfall data, generated by [pyraingen](https://github.com/crdykman/pyraingen). You can install pyraingen locally and calibrate it to available subdaily rainfall observations. Alternatively, you can call a remote pyraingen API and generate data based on publicly available data. We wil use the remote API in this example.

In [ ]:
rain_data_start = '2019-03-07'
rain_data_end = '2020-04-01'

replicates = get_rainfall_replicates(proj,None, rain_data_start, rain_data_end, 10, 
                                     600, # mean annual rainfall (mm)
                                     20,  # mean temperature (C)
                                     2)   # num_years
rainfall_data= replicates[catchment_name]
display(rainfall_data)

In [ ]:
aggregate_rainfall_data?

In [ ]:
rainfall_30min = aggregate_rainfall_data(rainfall_data,rain_data_start,rain_data_end)
rainfall_30min

**Note:** The pyraingen data includes 10 stochastic replicates. In the following examples, we will only use one replicate.


## Erosion - Daily simulation

Erosion processed are modelled cell-by-cell, timestep-by-timestep.

This can result in a large amount of output that in most cases is not directly usable.

 The library currently multiple ways to extract data from the model as it runs via customisable 'recorders'.


In [ ]:
# Grab a single rainfall sequence from the stochastic replicates
rain_seq = rainfall_30min.rainfall[:,9].to_pandas()
display(rain_seq)

### Setting up recorders

We want multiple gridded outputs:
* Total erosion for the year following the fire (year 1),
* Total erosion for the second year post fire
* Peak 30 min erosion for year 1 and for year 2

We also want daily timeseries at the subcatchment scale that we can later import into Source

In [ ]:
# Define the end of year 1 and the start of year 2
y1_end = rain_seq.index[len(rain_seq)//2]
y2_start = rain_seq.index[1+len(rain_seq)//2]

In [ ]:
recorders = dict(
    erosion_y1        = rusle.record_summary_grid('RUSLE',end_time=y1_end),              # Sum the 30min erosion until the end of year 1
    erosion_y2        = rusle.record_summary_grid('RUSLE',start_time=y2_start),          # Sum the 30min erosion from the start of year 2 to the end of the simulation
    peak_erosion_y1   = rusle.record_summary_grid('RUSLE',fn='max',end_time=y1_end),     # Find the peak 30min erosion rate from the start of the simulation to the end of year 1
    peak_erosion_y2   = rusle.record_summary_grid('RUSLE',fn='max',start_time=y2_start), # Find the peak 30min erosion rate from the start of year 2 to the end of the simulation
    erosion_daily_time_series = rusle.record_subcatchment_timeseries(proj,'RUSLE',agg_count=48)  # Record daily time series by aggregating every 48 time steps (30min)
)

# Run the simulations
## Erosion

In [ ]:
results = rusle.run_usle_simulation(proj,rain_seq,recorders=recorders)

In [ ]:
catchment_results = results[catchment_name]
catchment_results['erosion_daily_time_series']

In [ ]:
proj.plot_catchment_raster('Results', 'peak_erosion_y1')

In [ ]:
proj.plot_catchment_raster('Results', 'erosion_y1')

## Debris Flow

Debris flow operates on headwaters only and uses 12 minute rainfall _intensity_ data.


In [ ]:
from fire_impacts.sim import convert_rainfall_depth_to_intensity

In [ ]:
rainfall_intensity = convert_rainfall_depth_to_intensity(rainfall_data)
rainfall = aggregate_rainfall_data(rainfall_intensity,rain_data_start,rain_data_end,time_res='12min')
rainfall.rainfall

In [ ]:
rain_intensity_seq = rainfall.rainfall[:,9].to_pandas()

In [ ]:
df_results, df_ts = debris.debris_flow(
    proj,
    rain_intensity_seq,
    catchment=catchment_name
    )


In [ ]:
display(df_results)